# Inpainting Diagnostics

- 加载图像、mask 和 checkpoint
- 重建图像并计算 full / observed-only / missing-only PSNR
- 可视化整图、observed 区域、missing 区域的误差
- 对 elementwise mask 显示 coverage map
- 分析模型参数分布、空间位置和高权重点


In [ ]:
from pathlib import Path
import sys
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision.transforms as transforms
import yaml
from IPython.display import display
from PIL import Image

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "inpainting_train.py").exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / "gsplat") not in sys.path:
    sys.path.insert(0, str(repo_root / "gsplat"))

from gaussianimage_cholesky import GaussianImage_Cholesky
from inpainting_utils import (
    compute_coverage_map,
    compute_error_map,
    compute_inpainting_psnrs,
    compute_ms_ssim,
    compute_region_error_map,
    generate_mask,
    get_missing_mask,
)

plt.style.use("default")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"repo_root: {repo_root}")
print(f"device: {device}")
if device.type != "cuda":
    print("Warning: this renderer typically expects a CUDA-capable gsplat build for forward rendering.")

In [ ]:
checkpoint_path = repo_root / "checkpoints_inpainting" / "your_run" / "gaussian_model.pth.tar"
config_path = checkpoint_path.parent / "config.yaml"

# 如果 per-image config.yaml 没有 image_path，手动在这里指定。
image_path = None

# 如果训练时保存了 mask PNG，可以直接加载；否则会按 config + seed 重新生成。
use_saved_mask_if_available = False
mask_image_path = None

# 分析参数
topk_points = 20
hist_bins = 60
scatter_alpha = 0.35
scatter_size = 6

assert checkpoint_path.exists(), f"Checkpoint not found: {checkpoint_path}"
assert config_path.exists(), f"Config not found: {config_path}"

In [ ]:
def normalize_image_tensor(img_tensor, normalize_mode="auto"):
    img_tensor = img_tensor.float()
    if normalize_mode == "none":
        return img_tensor

    min_value = img_tensor.amin()
    max_value = img_tensor.amax()
    if normalize_mode == "auto":
        if min_value >= 0.0 and max_value <= 1.0:
            return img_tensor
        if min_value >= 0.0 and max_value <= 255.0:
            return img_tensor / 255.0
    if max_value > min_value:
        return (img_tensor - min_value) / (max_value - min_value)
    return torch.zeros_like(img_tensor)


def load_image_tensor(path, normalize_mode="auto"):
    tensor = transforms.ToTensor()(Image.open(path).convert("RGB")).unsqueeze(0)
    return normalize_image_tensor(tensor, normalize_mode=normalize_mode)


def load_yaml(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)


def set_all_seeds(seed):
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)


def load_mask_from_png(path, expected_channels):
    mask = transforms.ToTensor()(Image.open(path).convert("RGB")).unsqueeze(0).float()
    mask = (mask > 0.5).float()
    if expected_channels == 1:
        mask = mask[:, :1]
    return mask


def build_observed_mask(config, image_tensor, device):
    if mask_image_path is not None and Path(mask_image_path).exists():
        return load_mask_from_png(mask_image_path, image_tensor.shape[1]).to(device), "saved-png"

    inferred_mask_path = checkpoint_path.parent / f"{Path(image_path).stem}_mask.png"
    if use_saved_mask_if_available and inferred_mask_path.exists():
        return load_mask_from_png(inferred_mask_path, image_tensor.shape[1]).to(device), "saved-png"

    set_all_seeds(int(config.get("seed", 1)))
    mask = generate_mask(
        image_tensor.shape[-2],
        image_tensor.shape[-1],
        mask_type=config.get("mask_type", "random"),
        mask_ratio=float(config.get("mask_ratio", 0.5)),
        block_size=int(config.get("block_size", 64)),
        num_blocks=int(config.get("num_blocks", 4)),
        C=image_tensor.shape[1],
        device=device,
    )
    return mask, "regenerated-from-config"


def build_model_from_config(config, image_tensor, device):
    H, W = image_tensor.shape[-2], image_tensor.shape[-1]
    model = GaussianImage_Cholesky(
        loss_type="L2",
        opt_type="adan",
        num_points=int(config["num_points"]),
        H=H,
        W=W,
        BLOCK_H=16,
        BLOCK_W=16,
        device=device,
        lr=float(config.get("lr", 1e-3)),
        num_gabor=int(config.get("num_gabor", 2)),
        quantize=False,
    ).to(device)

    checkpoint = torch.load(checkpoint_path, map_location=device)
    model_dict = model.state_dict()
    filtered = {k: v for k, v in checkpoint.items() if k in model_dict}
    model_dict.update(filtered)
    model.load_state_dict(model_dict)
    model.eval()
    return model


def tensor_to_hwc(tensor):
    if tensor.dim() == 4:
        tensor = tensor.squeeze(0)
    tensor = tensor.detach().float().cpu()
    if tensor.shape[0] == 1:
        return tensor.squeeze(0).numpy()
    return tensor.permute(1, 2, 0).numpy()


def summarize_tensor(name, tensor):
    flat = tensor.detach().float().reshape(-1).cpu()
    return {
        "name": name,
        "shape": tuple(tensor.shape),
        "mean": float(flat.mean().item()),
        "std": float(flat.std(unbiased=False).item()),
        "min": float(flat.min().item()),
        "max": float(flat.max().item()),
    }


def effective_parameter_views(model):
    xy = model.get_xyz.detach().cpu()
    return {
        "xyz_tanh": xy,
        "cholesky_effective": model.get_cholesky_elements.detach().cpu(),
        "features_dc": model.get_features.detach().cpu(),
        "gabor_freqs": model.get_gabor_freqs.detach().cpu(),
        "gabor_weights_sigmoid": model.get_gabor_weights.detach().cpu(),
    }

In [ ]:
config = load_yaml(config_path)
if image_path is None:
    image_path = config.get("image_path", None)

if image_path is None:
    raise ValueError("This run config does not contain image_path. Please set image_path manually in the config cell.")

image_path = Path(image_path)
if not image_path.is_absolute():
    image_path = (repo_root / image_path).resolve()

gt_image = load_image_tensor(image_path, normalize_mode=config.get("normalize_mode", "auto")).to(device)
observed_mask, mask_source = build_observed_mask(config, gt_image, device)
missing_mask = get_missing_mask(observed_mask)
coverage_map = compute_coverage_map(observed_mask)
observed_image = gt_image * observed_mask

model = build_model_from_config(config, gt_image, device)
with torch.no_grad():
    reconstruction = model()["render"]

metrics = compute_inpainting_psnrs(reconstruction, gt_image, observed_mask)
ms_ssim_value = compute_ms_ssim(reconstruction, gt_image)

print(f"image_path: {image_path}")
print(f"mask_source: {mask_source}")
print(f"mask_type: {config.get('mask_type')} | mask_ratio: {config.get('mask_ratio')}")
print(f"PSNR(full): {metrics['psnr_full']:.4f}")
print(f"PSNR(observed-only): {metrics['psnr_observed']:.4f}")
print(f"PSNR(missing-only): {metrics['psnr_missing']:.4f}")
print(f"MS-SSIM(full): {ms_ssim_value:.6f}")

In [ ]:
full_error = compute_error_map(reconstruction, gt_image)
observed_error = compute_region_error_map(reconstruction, gt_image, observed_mask)
missing_error = compute_region_error_map(reconstruction, gt_image, missing_mask)

vis_items = [
    ("Ground Truth", tensor_to_hwc(gt_image)),
    ("Observed Image", tensor_to_hwc(observed_image)),
    ("Reconstruction", tensor_to_hwc(reconstruction)),
    ("Full Error", tensor_to_hwc((full_error / 0.5).clamp(0, 1))),
    ("Observed-only Error", tensor_to_hwc((observed_error / 0.5).clamp(0, 1))),
    ("Missing-only Error", tensor_to_hwc((missing_error / 0.5).clamp(0, 1))),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, (title, img) in zip(axes.flat, vis_items):
    if img.ndim == 2:
        ax.imshow(img, cmap="viridis", vmin=0.0, vmax=1.0)
    else:
        ax.imshow(np.clip(img, 0.0, 1.0))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

if observed_mask.shape[1] > 1:
    plt.figure(figsize=(6, 5))
    plt.imshow(tensor_to_hwc(coverage_map), cmap="magma", vmin=0.0, vmax=1.0)
    plt.title("Coverage Map (elementwise mask)")
    plt.colorbar()
    plt.axis("off")
    plt.show()

In [ ]:
param_views = effective_parameter_views(model)
stats_rows = [summarize_tensor(name, tensor) for name, tensor in param_views.items()]
stats_df = pd.DataFrame(stats_rows)
display(stats_df)

num_points = model.init_num_points
num_gabor = model.get_num_gabor
point_strength = model.get_gabor_weights.detach().cpu().view(num_points, num_gabor).mean(dim=1)
topk = min(topk_points, num_points)
top_idx = torch.topk(point_strength, topk).indices
xy_top = model.get_xyz.detach().cpu()[top_idx]
top_points_df = pd.DataFrame({
    "point_index": top_idx.numpy(),
    "mean_gabor_weight": point_strength[top_idx].numpy(),
    "x": xy_top[:, 0].numpy(),
    "y": xy_top[:, 1].numpy(),
})
display(top_points_df)

In [ ]:
plot_groups = {
    "xyz_tanh": param_views["xyz_tanh"].reshape(-1).numpy(),
    "cholesky_effective": param_views["cholesky_effective"].reshape(-1).numpy(),
    "features_dc": param_views["features_dc"].reshape(-1).numpy(),
    "gabor_freqs": param_views["gabor_freqs"].reshape(-1).numpy(),
    "gabor_weights_sigmoid": param_views["gabor_weights_sigmoid"].reshape(-1).numpy(),
}

fig, axes = plt.subplots(len(plot_groups), 2, figsize=(14, 4 * len(plot_groups)))
for row_idx, (name, values) in enumerate(plot_groups.items()):
    axes[row_idx, 0].hist(values, bins=hist_bins, color="tab:blue", alpha=0.85)
    axes[row_idx, 0].set_title(f"{name} histogram")
    axes[row_idx, 0].grid(alpha=0.2)

    axes[row_idx, 1].boxplot(values, vert=False)
    axes[row_idx, 1].set_title(f"{name} boxplot")
    axes[row_idx, 1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
xy = model.get_xyz.detach().cpu()
x_px = ((xy[:, 0] + 1.0) * 0.5 * (gt_image.shape[-1] - 1)).numpy()
y_px = ((xy[:, 1] + 1.0) * 0.5 * (gt_image.shape[-2] - 1)).numpy()
strength = point_strength.numpy()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].imshow(np.clip(tensor_to_hwc(gt_image), 0.0, 1.0))
axes[0].scatter(x_px, y_px, s=scatter_size, c="cyan", alpha=scatter_alpha)
axes[0].set_title("Gaussian / Gabor Centers")
axes[0].axis("off")

axes[1].imshow(np.clip(tensor_to_hwc(gt_image), 0.0, 1.0))
scatter = axes[1].scatter(x_px, y_px, s=scatter_size, c=strength, cmap="inferno", alpha=scatter_alpha)
axes[1].set_title("Centers Colored by Mean Gabor Weight")
axes[1].axis("off")
fig.colorbar(scatter, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()